# 06 — LightGBM: M5 Walmart Demand Intelligence

**Purpose:** Train LightGBM on the v2 feature set (same 39 features as 05c).
Structure mirrors `05c_xgboost_v2.ipynb` exactly — same fold boundaries, same
evaluation functions, same three-tier metric system, same post-hoc inference
rules. The only variable that changes is the model class.

**Why this comparison is valid:** Both models train on identical inputs
(`features_train_v2.parquet`, `features_val_v2.parquet`), identical fold
boundaries, and identical post-hoc inference rules (holiday suppression +
isotonic calibration loaded from 05c pkl files — not re-fit). Any performance
difference is attributable to the model, not the pipeline.

**Inputs:**
- `../data/processed/features/features_train_v2.parquet`
- `../data/processed/features/features_val_v2.parquet`
- `../data/processed/features/feature_cols_v2.pkl`
- `../data/processed/calibration/isotonic_calibrator_fold2.pkl`  ← from 05c, not re-fit
- `../data/processed/calibration/train_zero_rate_fold2.pkl`       ← from 05c, not re-fit
- `../data/processed/calibration/pipeline_decisions_fold2.pkl`    ← CALIBRATION_ADOPTED flag

**Outputs:**
- `../data/processed/models/lgbm_model_fold2.pkl`
- `../data/processed/predictions/lgbm_predictions_fold2.parquet`
- `../data/processed/calibration/lgbm_best_params.pkl`

| Fold | Role |
|---|---|
| Fold 1 | Exploratory — default LightGBM params on v2 features, establishes untuned floor |
| Fold 2 | Primary tuning — Optuna 50 trials, params frozen on convergence |
| Fold 3 | **Never touched in this notebook** |

> **Calibration note:** The isotonic calibrator and zero-rate lookup are loaded
> directly from 05c pkl files — not re-fit on LightGBM residuals. The calibration
> is indexed on series zero-rate (a data property, not a model property). Using the
> same calibrator ensures the Fold 2 XGBoost vs LightGBM comparison is apples-to-apples.

> **Demand proxy reminder:** Observed sales proxy true latent demand. Zero sales
> may reflect a stockout or genuine absence. All outputs are demand approximations.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
import pickle
import os
import time
import warnings
import subprocess

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.isotonic import IsotonicRegression

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams['figure.figsize'] = (14, 5)
np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────
PROCESSED_DIR   = '../data/processed'
FEATURES_DIR    = f'{PROCESSED_DIR}/features'
CALIBRATION_DIR = f'{PROCESSED_DIR}/calibration'
MODELS_DIR      = f'{PROCESSED_DIR}/models'
PREDICTIONS_DIR = f'{PROCESSED_DIR}/predictions'
REP_SERIES      = 'FOODS_3_163_CA_3_validation'

# ── v2 file paths ──────────────────────────────────────────────────────────
FEATURES_TRAIN_V2 = f'{FEATURES_DIR}/features_train_v2.parquet'
FEATURES_VAL_V2   = f'{FEATURES_DIR}/features_val_v2.parquet'
FEATURE_COLS_V2   = f'{FEATURES_DIR}/feature_cols_v2.pkl'

# ── Walk-forward fold boundaries (identical to 05c) ────────────────────────
FOLDS = {
    'fold_1': {
        'train_start':   '2011-02-01',
        'monitor_start': '2012-12-02',
        'train_end':     '2013-01-31',
        'val_start':     '2013-02-01',
        'val_end':       '2014-01-31',
    },
    'fold_2': {
        'train_start':   '2011-02-01',
        'monitor_start': '2013-12-02',
        'train_end':     '2014-01-31',
        'val_start':     '2014-02-01',
        'val_end':       '2015-01-31',
    },
    'fold_3': {
        'train_start':   '2011-02-01',
        'monitor_start': '2014-12-02',
        'train_end':     '2015-01-31',
        'val_start':     '2015-02-01',
        'val_end':       '2016-01-31',
    },
}

# ── LightGBM constants ─────────────────────────────────────────────────────
EARLY_STOPPING_ROUNDS = 50
OPTUNA_TRIALS         = 25
TARGET_COL            = 'target'

# Default LightGBM params — used in Section 4 as the Fold 1 untuned baseline.
# These establish the floor before Optuna runs on Fold 2.
LGBM_DEFAULT_PARAMS = {
    'num_leaves':        127,
    'learning_rate':     0.05,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'min_child_samples': 20,
    'reg_alpha':         0.0,
    'reg_lambda':        0.0,
    'objective':         'regression',
    'metric':            'rmse',
    'verbosity':         -1,
    'random_state':      42,
    'num_threads':       -1,
}


# BEST_PARAMS_LGBM — populated in Section 5 after Optuna on Fold 2.
# Written here as a named constant so the freeze is visible and auditable.
# DO NOT modify after Section 5 is complete.
BEST_PARAMS_LGBM = None  # replaced after tuning


# ── Evaluation functions (identical to 05c) ────────────────────────────────
def eval_log_scale(y_true_log, y_pred_log, label):
    """
    Tier 1 / Tier 2 metric. Operates in log1p space on non-zero actual rows.
    Non-zero filter: rows where actual=0 are structural gap rows.
    Bias reported alongside accuracy — systematic underprediction causes stockouts.
    """
    mask = y_true_log > 0
    n    = mask.sum()
    rmse = np.sqrt(mean_squared_error(y_true_log[mask], y_pred_log[mask]))
    mae  = mean_absolute_error(y_true_log[mask], y_pred_log[mask])
    bias = float(np.mean(y_pred_log[mask] - y_true_log[mask]))

    print(f'{label}  [non-zero rows: {n:,}]')
    print(f'  log-RMSE: {rmse:.4f}')
    print(f'  log-MAE:  {mae:.4f}')
    print(f'  Bias:     {bias:+.4f}  (+ = overpredict, − = underpredict)')
    print()
    return {'label': label, 'log_rmse': rmse, 'log_mae': mae, 'bias': bias, 'n': n}


def eval_rep_series_monthly(predictions_df, actuals_df, label):
    """
    Tier 3 metric. FOODS_3_163_CA_3 only, monthly revenue, non-zero months.
    The only slice directly comparable to SARIMA (22.22%) and Prophet (24.25%).
    Never call on any other slice or granularity.
    """
    pred = predictions_df[predictions_df['id'] == REP_SERIES].copy()
    act  = actuals_df[actuals_df['id'] == REP_SERIES].copy()

    pred['month'] = pd.to_datetime(pred['date']).dt.to_period('M')
    act['month']  = pd.to_datetime(act['date']).dt.to_period('M')

    pred_monthly = pred.groupby('month')['yhat'].sum()
    act['revenue'] = act['units_sold'] * act['sell_price'].fillna(0)
    act_monthly  = act.groupby('month')['revenue'].sum()

    combined = pd.DataFrame({'actual': act_monthly, 'predicted': pred_monthly}).dropna()
    nonzero  = combined[combined['actual'] > 0]
    mape     = float(np.mean(np.abs((nonzero['actual'] - nonzero['predicted']) / nonzero['actual'])) * 100)

    print(f'{label}  [representative series, monthly revenue, non-zero months: {len(nonzero)}]')
    print(f'  MAPE: {mape:.2f}%')
    print(f'  Baseline — SARIMA: 22.22%  |  Prophet: 24.25%')
    print()
    return {'label': label, 'mape': mape, 'n_months': len(nonzero)}


def eval_quantiles(y_true, preds_dict, label):
    """
    Tier 4 metric. Pinball loss + empirical coverage per quantile.
    preds_dict: {0.50: array, 0.80: array, 0.95: array, 0.99: array}
    All arrays in original unit space (post-expm1).
    """
    print(f'{label}')
    print(f'  {"Quantile":<12} {"Pinball Loss":>14} {"Coverage":>10} {"Target":>10}')
    print(f'  {"─"*50}')
    results = {}
    for q, y_pred in sorted(preds_dict.items()):
        errors   = y_true - y_pred
        pinball  = float(np.mean(np.where(errors >= 0, q * errors, (q - 1) * errors)))
        coverage = float(np.mean(y_true <= y_pred) * 100)
        results[q] = {'pinball': pinball, 'coverage': coverage}
        print(f'  q{int(q*100):<11} {pinball:>14.4f} {coverage:>9.1f}% {q*100:>9.0f}%')
    print()
    return results


def get_gpu_stats():
    """Return GPU temperature, utilization, and memory usage as a string."""
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=temperature.gpu,utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader'],
            capture_output=True, text=True
        )
        temp, util, mem_used, mem_total = result.stdout.strip().split(',')
        return f'GPU: {temp.strip()}°C | {util.strip()} util | {mem_used.strip()} / {mem_total.strip()}'
    except:
        return 'GPU stats unavailable'


print('Setup complete.')
print(f'LightGBM version: {lgb.__version__}')
print(f'BEST_PARAMS_LGBM = None — will be populated after Fold 2 Optuna.')
print()
print(f'{get_gpu_stats()}')

Setup complete.
LightGBM version: 4.6.0
BEST_PARAMS_LGBM = None — will be populated after Fold 2 Optuna.

GPU: 51°C | 0 % util | 169 MiB / 6144 MiB


Constants and evaluation functions defined. Three things to note:

**`LGBM_DEFAULT_PARAMS` is used in Section 4 only** — to establish how default
LightGBM parameters perform on v2 features before Optuna runs. This is the
untuned floor.

**`BEST_PARAMS_LGBM = None` is intentional.** Replaced after Fold 2 tuning in
Section 5. If still `None` when Section 6 runs, something went wrong.

**Fold 3 boundary is never crossed in this notebook.** The `fold_3` entry in
`FOLDS` is defined for reference only — no cell below uses it.

## 2. Load v2 Feature Matrix

In [3]:
train = pd.read_parquet(FEATURES_TRAIN_V2)
val   = pd.read_parquet(FEATURES_VAL_V2)

with open(FEATURE_COLS_V2, 'rb') as f:
    FEATURE_COLS = pickle.load(f)

print(f'Train : {len(train):>12,} rows  |  {train["date"].min().date()} → {train["date"].max().date()}')
print(f'Val   : {len(val):>12,} rows  |  {val["date"].min().date()} → {val["date"].max().date()}')
print(f'Features : {len(FEATURE_COLS)}  (v1 had 34 — delta: +{len(FEATURE_COLS) - 34})')
print(f'Series   : {train["id"].nunique():,}')
print()

# Feature list confirmation
print('v2 feature list:')
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'  {i:>2}. {col}')
print()

# Leakage assertion — asserted, not assumed
assert train['date'].max() < pd.Timestamp(val['date'].min()), \
    'LEAKAGE: training data overlaps with validation window'
print('Leakage check passed. ✓')
print()

# Null audit
null_counts = train[FEATURE_COLS].isna().sum()
null_counts = null_counts[null_counts > 0]
print('Null counts per feature (training set):')
for col, n in null_counts.items():
    print(f'  {col:<32} {n:>10,}  ({n/len(train)*100:.1f}%)')
print()

lag_cols = [c for c in FEATURE_COLS if c.startswith('lag_')]
gap_mask = train[lag_cols].isna().all(axis=1)
print(f'Rows with all lags null: {gap_mask.sum():,}  ({gap_mask.mean()*100:.1f}%)')

Train :   28,699,814 rows  |  2011-02-02 → 2015-01-31
Val   :   11,128,850 rows  |  2015-02-01 → 2016-01-31
Features : 39  (v1 had 34 — delta: +5)
Series   : 30,490

v2 feature list:
   1. day_of_week
   2. day_of_month
   3. week_of_year
   4. month_num
   5. is_weekend
   6. is_month_start
   7. is_month_end
   8. is_event
   9. is_closed_holiday
  10. days_to_closed_holiday
  11. days_to_holiday_proximity
  12. preholiday_x_cat
  13. is_snap
  14. snap_day_of_cycle
  15. is_snap_peak
  16. sell_price
  17. price_change_pct
  18. price_drop
  19. price_increase
  20. price_rel_28
  21. price_vs_item_mean
  22. price_percentile_52w
  23. lag_1
  24. lag_7
  25. lag_14
  26. lag_28
  27. rolling_mean_7
  28. rolling_mean_28
  29. rolling_std_7
  30. store_rolling_7
  31. store_rolling_28
  32. dept_rolling_7
  33. dept_rolling_28
  34. store_id_enc
  35. item_id_enc
  36. dept_id_enc
  37. cat_id_enc
  38. state_id_enc
  39. weekday_enc

Leakage check passed. ✓

Null counts per feature

All 30,490 series present. Schema confirmed against `feature_cols_v2.pkl`.
Leakage assertion passed.

LightGBM handles null values natively via histogram binning — no imputation
required. Null lags and null price features are assigned to a separate bin.

## 3. Walk-Forward CV Setup

In [4]:
def get_fold_data(fold):
    """Split into train, monitor, and val sets for a given fold."""
    f = FOLDS[fold]

    train_df   = train[(train['date'] >= f['train_start']) &
                       (train['date'] <  f['monitor_start'])].copy()
    monitor_df = train[(train['date'] >= f['monitor_start']) &
                       (train['date'] <= f['train_end'])].copy()

    # Fold 3 val lives in the val parquet — all other folds are within train
    source = val if fold == 'fold_3' else train
    val_df = source[(source['date'] >= f['val_start']) &
                    (source['date'] <= f['val_end'])].copy()

    return train_df, monitor_df, val_df


print(f'{"Fold":<8} {"Train rows":>12} {"Monitor rows":>14} {"Val rows":>12} {"Train window":<28} {"Val window"}')
print('─' * 95)

for fold, f in FOLDS.items():
    tr, mo, va = get_fold_data(fold)
    print(
        f'{fold:<8} {len(tr):>12,} {len(mo):>14,} {len(va):>12,} '
        f'{f["train_start"]} → {f["train_end"]}   '
        f'{f["val_start"]} → {f["val_end"]}'
    )

Fold       Train rows   Monitor rows     Val rows Train window                 Val window
───────────────────────────────────────────────────────────────────────────────────────────────
fold_1     10,803,295      1,138,898    7,752,753 2011-02-01 → 2013-01-31   2013-02-01 → 2014-01-31
fold_2     18,360,368      1,334,578    9,004,868 2011-02-01 → 2014-01-31   2014-02-01 → 2015-01-31
fold_3     27,140,570      1,559,244   11,128,850 2011-02-01 → 2015-01-31   2015-02-01 → 2016-01-31


Row counts match 05c exactly — fold boundaries and train/val parquet contents
are identical. The v2 feature columns are wider but the row structure is unchanged.

Walk-forward expanding windows are the only valid evaluation strategy in a
time-series context — k-fold would require training on future data to predict
the past.

## 4. Hyperparameter Tuning — Fold 1 (Optuna)

Exploratory tuning on Fold 1. Wide search space, 25 trials. Objective is to
learn which parameters matter and what ranges work on the v2 feature set —
not to find the final configuration. Fold 2 is the primary tuning fold where
params get frozen.

Results inform the tighter Fold 2 search space in Section 5. Every trial
prints as it completes so you can watch Optuna converge in real time.

In [5]:
tr1, mo1, va1 = get_fold_data('fold_1')

X_tr1 = tr1[FEATURE_COLS]
y_tr1 = tr1[TARGET_COL]
X_mo1 = mo1[FEATURE_COLS]
y_mo1 = mo1[TARGET_COL]
X_va1 = va1[FEATURE_COLS]
y_va1 = va1[TARGET_COL].values

# ── Build datasets ONCE outside the objective ──────────────────────────────
print('Building LightGBM datasets...')
dtrain1 = lgb.Dataset(X_tr1, label=y_tr1, feature_name=list(FEATURE_COLS), free_raw_data=False)
dtrain1.construct()
dmon1   = lgb.Dataset(X_mo1, label=y_mo1, reference=dtrain1, free_raw_data=False)
dmon1.construct()
print('Datasets ready.')
print()


def make_objective_f1(dtrain, dmon, X_val, y_val_log):
    def objective(trial):
        params = {
            'num_leaves':        trial.suggest_int('num_leaves', 64, 512),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 5.0),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 5.0),
            'objective':         'regression',
            'metric':            'rmse',
            'verbosity':         -1,
            'random_state':      42,
            'num_threads':       -1,
        }

        model = lgb.train(
            params=params,
            train_set=dtrain,
            num_boost_round=1500,
            valid_sets=[dmon],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        trial.set_user_attr('best_trees', model.best_iteration)
        y_pred_log = model.predict(X_val)
        mask       = y_val_log > 0
        rmse       = np.sqrt(mean_squared_error(y_val_log[mask], y_pred_log[mask]))
        return rmse

    return objective


def optuna_callback(study, trial):
    trees = trial.user_attrs.get('best_trees', '?')
    print(
        f'  Trial {trial.number:>3} | '
        f'log-RMSE: {trial.value:.4f} | '
        f'best: {study.best_value:.4f} | '
        f'trees={trees:>4} | '
        f'leaves={trial.params["num_leaves"]:>3} '
        f'lr={trial.params["learning_rate"]:.3f} '
        f'ff={trial.params["feature_fraction"]:.2f} '
        f'bf={trial.params["bagging_fraction"]:.2f} '
        f'mcs={trial.params["min_child_samples"]:>3} '
        f'α={trial.params["reg_alpha"]:.2f} '
        f'λ={trial.params["reg_lambda"]:.2f}'
    )
    if trial.number % 5 == 0:
        print(f'  {get_gpu_stats()}')


print('Fold 1 — Optuna tuning started...')
print(f'  {get_gpu_stats()}')
print()

study_f1 = optuna.create_study(direction='minimize')
study_f1.optimize(
    make_objective_f1(dtrain1, dmon1, X_va1, y_va1),
    n_trials=25,
    callbacks=[optuna_callback],
)

print()
print(f'  {get_gpu_stats()}')
print()
print(f'Fold 1 best log-RMSE : {study_f1.best_value:.4f}')
print()
print('Best params:')
for k, v in study_f1.best_params.items():
    print(f'  {k:<22} {v}')
print()
print('Use these results to inform the Fold 2 search space in Section 5.')
print('Tighten ranges around the values Fold 1 converged to.')
print('If num_leaves consistently hit the ceiling (512), raise it.')
print('If reg_alpha/lambda stayed near 0, reduce the upper bound.')

Building LightGBM datasets...
Datasets ready.

Fold 1 — Optuna tuning started...
  GPU: 48°C | 0 % util | 169 MiB / 6144 MiB

  Trial   0 | log-RMSE: 0.5798 | best: 0.5798 | trees= 463 | leaves=167 lr=0.062 ff=0.73 bf=0.67 mcs= 11 α=4.48 λ=3.76
  GPU: 59°C | 0 % util | 169 MiB / 6144 MiB
  Trial   1 | log-RMSE: 0.5791 | best: 0.5791 | trees= 226 | leaves=404 lr=0.105 ff=0.94 bf=0.90 mcs= 53 α=2.05 λ=2.47
  Trial   2 | log-RMSE: 0.5800 | best: 0.5791 | trees= 213 | leaves=381 lr=0.119 ff=0.74 bf=0.98 mcs= 37 α=0.14 λ=4.01
  Trial   3 | log-RMSE: 0.5793 | best: 0.5791 | trees= 824 | leaves=160 lr=0.062 ff=0.57 bf=0.72 mcs= 76 α=1.78 λ=4.86
  Trial   4 | log-RMSE: 0.5790 | best: 0.5790 | trees= 507 | leaves=465 lr=0.040 ff=0.62 bf=0.56 mcs= 96 α=0.76 λ=1.35
  Trial   5 | log-RMSE: 0.5793 | best: 0.5790 | trees= 806 | leaves=306 lr=0.037 ff=0.74 bf=0.66 mcs= 32 α=2.05 λ=3.73
  GPU: 59°C | 0 % util | 169 MiB / 6144 MiB
  Trial   6 | log-RMSE: 0.5792 | best: 0.5790 | trees= 714 | leaves=243 

[W 2026-06-02 17:08:15,724] Trial 22 failed with parameters: {'num_leaves': 366, 'learning_rate': 0.023783538016124026, 'feature_fraction': 0.6628998617714108, 'bagging_fraction': 0.839288415459002, 'bagging_freq': 6, 'min_child_samples': 89, 'reg_alpha': 2.7805565861022425, 'reg_lambda': 1.3854812679619193} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Zach\AppData\Local\Temp\ipykernel_22968\4171690283.py", line 38, in objective
    model = lgb.train(
        params=params,
    ...<6 lines>...
        ],
    )
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\engine.py", line 322, in train
    booster.update(fobj=fobj)
    ~~~~~~~~~~~~~~^^^^^^^^^^^
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-package

KeyboardInterrupt: 

Fold 1 exploratory run establishes the pre-tuning floor on v2 features.
The bias direction should be negative (systematic underprediction), consistent
with 05c. If it flipped positive, a new feature is leaking future signal.

Review convergence before proceeding: tighten the Fold 2 search space in
Section 5 around the ranges where Fold 1 found its best trials.

In [5]:
# ═══════════════════════════════════════════════════════════════
# TWEEDIE FOLD 1 — Hyperparameter Search
# ═══════════════════════════════════════════════════════════════

tr1, mo1, va1 = get_fold_data('fold_1')

X_tr1_tw = tr1[FEATURE_COLS]
y_tr1_tw = np.expm1(tr1[TARGET_COL])   # ← raw units, NOT log
X_mo1_tw = mo1[FEATURE_COLS]
y_mo1_tw = np.expm1(mo1[TARGET_COL])   # ← raw units
X_va1_tw = va1[FEATURE_COLS]
y_va1_tw = np.expm1(va1[TARGET_COL].values)  # ← raw units for eval

print('Building Tweedie Fold 1 datasets...')
dtrain1_tw = lgb.Dataset(X_tr1_tw, label=y_tr1_tw, feature_name=list(FEATURE_COLS), free_raw_data=False)
dtrain1_tw.construct()
dmon1_tw   = lgb.Dataset(X_mo1_tw, label=y_mo1_tw, reference=dtrain1_tw, free_raw_data=False)
dmon1_tw.construct()
print('Datasets ready.')
print()


def make_objective_tweedie_f1(dtrain, dmon, X_val, y_val_units):
    def objective(trial):
        params = {
            'num_leaves':              trial.suggest_int('num_leaves', 64, 512),
            'learning_rate':           trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'feature_fraction':        trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction':        trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq':            trial.suggest_int('bagging_freq', 1, 7),
            'min_child_samples':       trial.suggest_int('min_child_samples', 10, 100),
            'reg_alpha':               trial.suggest_float('reg_alpha', 0.0, 5.0),
            'reg_lambda':              trial.suggest_float('reg_lambda', 0.0, 5.0),
            'tweedie_variance_power':  trial.suggest_float('tweedie_variance_power', 1.0, 1.9),
            'objective':               'tweedie',
            'metric':                  'tweedie',
            'verbosity':               -1,
            'random_state':            42,
            'num_threads':             -1,
        }

        model = lgb.train(
            params=params,
            train_set=dtrain,
            num_boost_round=1500,
            valid_sets=[dmon],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        trial.set_user_attr('best_trees', model.best_iteration)

        # Evaluate in raw unit space so it's comparable to RMSE model later
        y_pred_units = model.predict(X_val)
        mask         = y_val_units > 0
        rmse         = np.sqrt(mean_squared_error(y_val_units[mask], y_pred_units[mask]))
        return rmse

    return objective


def optuna_callback_tw(study, trial):
    trees = trial.user_attrs.get('best_trees', '?')
    print(
        f'  Trial {trial.number:>3} | '
        f'unit-RMSE: {trial.value:.2f} | '
        f'best: {study.best_value:.2f} | '
        f'trees={trees:>4} | '
        f'leaves={trial.params["num_leaves"]:>3} '
        f'lr={trial.params["learning_rate"]:.3f} '
        f'ff={trial.params["feature_fraction"]:.2f} '
        f'bf={trial.params["bagging_fraction"]:.2f} '
        f'mcs={trial.params["min_child_samples"]:>3} '
        f'tvp={trial.params["tweedie_variance_power"]:.2f} '
        f'α={trial.params["reg_alpha"]:.2f} '
        f'λ={trial.params["reg_lambda"]:.2f}'
    )
    if trial.number % 5 == 0:
        print(f'  {get_gpu_stats()}')


print('Tweedie Fold 1 — Optuna tuning started...')
print(f'  {get_gpu_stats()}')
print()

study_tw_f1 = optuna.create_study(direction='minimize')
study_tw_f1.optimize(
    make_objective_tweedie_f1(dtrain1_tw, dmon1_tw, X_va1_tw, y_va1_tw),
    n_trials=15,
    callbacks=[optuna_callback_tw],
)

print()
print(f'Tweedie Fold 1 best unit-RMSE : {study_tw_f1.best_value:.2f}')
print()
print('Best params:')
for k, v in study_tw_f1.best_params.items():
    print(f'  {k:<26} {v}')
print()
print('Use these to tighten Tweedie Fold 2 search space below.')

Building Tweedie Fold 1 datasets...
Datasets ready.

Tweedie Fold 1 — Optuna tuning started...
  GPU: 53°C | 0 % util | 169 MiB / 6144 MiB

  Trial   0 | unit-RMSE: 3.74 | best: 3.74 | trees= 120 | leaves=363 lr=0.092 ff=0.75 bf=0.78 mcs= 43 tvp=1.58 α=4.43 λ=4.93
  GPU: 58°C | 0 % util | 169 MiB / 6144 MiB
  Trial   1 | unit-RMSE: 3.74 | best: 3.74 | trees= 902 | leaves=262 lr=0.010 ff=0.51 bf=0.85 mcs= 79 tvp=1.59 α=0.00 λ=4.80
  Trial   2 | unit-RMSE: 3.72 | best: 3.72 | trees= 528 | leaves=161 lr=0.034 ff=0.55 bf=0.87 mcs= 46 tvp=1.40 α=3.68 λ=2.26
  Trial   3 | unit-RMSE: 3.71 | best: 3.71 | trees= 485 | leaves=225 lr=0.022 ff=0.93 bf=0.82 mcs= 47 tvp=1.07 α=3.71 λ=2.66
  Trial   4 | unit-RMSE: 3.78 | best: 3.71 | trees= 199 | leaves=505 lr=0.119 ff=0.98 bf=0.56 mcs= 93 tvp=1.19 α=3.66 λ=1.21
  Trial   5 | unit-RMSE: 3.75 | best: 3.71 | trees= 233 | leaves=412 lr=0.046 ff=0.86 bf=0.96 mcs= 49 tvp=1.54 α=4.16 λ=0.65
  GPU: 59°C | 0 % util | 169 MiB / 6144 MiB
  Trial   6 | unit-RMS

## 5. Hyperparameter Tuning — Fold 2 (Optuna)

Primary tuning fold. 50 trials, Bayesian optimization, objective = log-RMSE
on the Fold 2 val window. Search space tightened based on Fold 1 convergence.
A warm-start trial enqueues Fold 1's best params as the first candidate so
Optuna has a strong starting point.

When satisfied with convergence (no improvement in final 20 trials), best params
are written as `BEST_PARAMS_LGBM` and frozen. Do not modify after this section.

In [20]:
tr2, mo2, va2 = get_fold_data('fold_2')

X_tr2 = tr2[FEATURE_COLS]
y_tr2 = tr2[TARGET_COL]
X_mo2 = mo2[FEATURE_COLS]
y_mo2 = mo2[TARGET_COL]
X_va2 = va2[FEATURE_COLS]
y_va2 = va2[TARGET_COL].values

# ── Build datasets ONCE outside the objective ──────────────────────────────
print('Building LightGBM datasets...')
dtrain2 = lgb.Dataset(X_tr2, label=y_tr2, feature_name=list(FEATURE_COLS), free_raw_data=False)
dtrain2.construct()
dmon2   = lgb.Dataset(X_mo2, label=y_mo2, reference=dtrain2, free_raw_data=False)
dmon2.construct()
print('Datasets ready.')
print()

# ── Fold 1 best params (Trial 13, log-RMSE 0.5783) ────────────────────────
FOLD1_BEST = {
    'num_leaves':        446,
    'learning_rate':     0.020,
    'feature_fraction':  0.52,
    'bagging_fraction':  0.81,
    'bagging_freq':      5,
    'min_child_samples': 82,
    'reg_alpha':         3.33,
    'reg_lambda':        0.02,
}


def make_objective_fold2(dtrain, dmon, X_val, y_val_log):
    def objective(trial):
        params = {
            'num_leaves':        trial.suggest_int('num_leaves', 300, 600),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.04, log=True),
            'feature_fraction':  trial.suggest_float('feature_fraction', 0.50, 0.75),
            'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.70, 0.95),
            'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
            'min_child_samples': trial.suggest_int('min_child_samples', 60, 110),
            'reg_alpha':         trial.suggest_float('reg_alpha', 2.0, 5.0),
            'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.5),
            'objective':         'regression',
            'metric':            'rmse',
            'verbosity':         -1,
            'random_state':      42,
            'num_threads':       -1,
        }

        model = lgb.train(
            params=params,
            train_set=dtrain,
            num_boost_round=2000,
            valid_sets=[dmon],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        trial.set_user_attr('best_trees', model.best_iteration)
        y_pred_log = model.predict(X_val)
        mask       = y_val_log > 0
        rmse       = np.sqrt(mean_squared_error(y_val_log[mask], y_pred_log[mask]))
        return rmse

    return objective


def optuna_callback(study, trial):
    trees = trial.user_attrs.get('best_trees', '?')
    print(
        f'  Trial {trial.number:>3} | '
        f'log-RMSE: {trial.value:.4f} | '
        f'best: {study.best_value:.4f} | '
        f'trees={trees:>4} | '
        f'leaves={trial.params["num_leaves"]:>3} '
        f'lr={trial.params["learning_rate"]:.3f} '
        f'ff={trial.params["feature_fraction"]:.2f} '
        f'bf={trial.params["bagging_fraction"]:.2f} '
        f'mcs={trial.params["min_child_samples"]:>3} '
        f'α={trial.params["reg_alpha"]:.2f} '
        f'λ={trial.params["reg_lambda"]:.2f}'
    )
    if trial.number % 5 == 0:
        print(f'  {get_gpu_stats()}')


print('Fold 2 — Optuna tuning started...')
print(f'  {get_gpu_stats()}')
print()

study_fold2 = optuna.create_study(direction='minimize')
study_fold2.enqueue_trial(FOLD1_BEST)

study_fold2.optimize(
    make_objective_fold2(dtrain2, dmon2, X_va2, y_va2),
    n_trials=OPTUNA_TRIALS,
    callbacks=[optuna_callback],
)

print()
print(f'  {get_gpu_stats()}')
print()
print(f'Fold 2 best log-RMSE : {study_fold2.best_value:.4f}')
print(f'XGBoost v2 reference : 0.5764')
print(f'Delta                : {study_fold2.best_value - 0.5764:+.4f}')
print()
print('Best params:')
for k, v in study_fold2.best_params.items():
    print(f'  {k:<22} {v}')

Building LightGBM datasets...
Datasets ready.

Fold 2 — Optuna tuning started...
  GPU: 37°C | 0 % util | 0 MiB / 6144 MiB

  Trial   0 | log-RMSE: 0.5771 | best: 0.5771 | trees=1989 | leaves=446 lr=0.020 ff=0.52 bf=0.81 mcs= 82 α=3.33 λ=0.02
  GPU: 57°C | 0 % util | 0 MiB / 6144 MiB
  Trial   1 | log-RMSE: 0.5772 | best: 0.5771 | trees=1185 | leaves=529 lr=0.033 ff=0.62 bf=0.73 mcs= 86 α=3.17 λ=1.42
  Trial   2 | log-RMSE: 0.5772 | best: 0.5771 | trees=1996 | leaves=497 lr=0.011 ff=0.72 bf=0.78 mcs= 85 α=4.19 λ=0.81
  Trial   3 | log-RMSE: 0.5769 | best: 0.5769 | trees=1976 | leaves=543 lr=0.022 ff=0.53 bf=0.88 mcs= 98 α=2.28 λ=1.19
  Trial   4 | log-RMSE: 0.5776 | best: 0.5769 | trees=1990 | leaves=311 lr=0.015 ff=0.55 bf=0.89 mcs= 75 α=3.52 λ=1.24
  Trial   5 | log-RMSE: 0.5775 | best: 0.5769 | trees=1996 | leaves=445 lr=0.011 ff=0.53 bf=0.72 mcs= 62 α=4.38 λ=0.69
  GPU: 59°C | 0 % util | 0 MiB / 6144 MiB
  Trial   6 | log-RMSE: 0.5767 | best: 0.5767 | trees=1768 | leaves=458 lr=0.0

[W 2026-06-03 15:28:01,407] Trial 29 failed with parameters: {'num_leaves': 562, 'learning_rate': 0.0176322234815884, 'feature_fraction': 0.5558161588250131, 'bagging_fraction': 0.8766303611862594, 'bagging_freq': 2, 'min_child_samples': 80, 'reg_alpha': 3.5141884937200287, 'reg_lambda': 1.4596693083550718} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Zach\AppData\Local\Temp\ipykernel_17460\1057449665.py", line 50, in objective
    model = lgb.train(
        params=params,
    ...<6 lines>...
        ],
    )
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\engine.py", line 322, in train
    booster.update(fobj=fobj)
    ~~~~~~~~~~~~~~^^^^^^^^^^^
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages

KeyboardInterrupt: 

In [6]:
# ═══════════════════════════════════════════════════════════════
# TWEEDIE FOLD 2 — Refined Hyperparameter Search
# ═══════════════════════════════════════════════════════════════

tr2, mo2, va2 = get_fold_data('fold_2')

X_tr2_tw = tr2[FEATURE_COLS]
y_tr2_tw = np.expm1(tr2[TARGET_COL])
X_mo2_tw = mo2[FEATURE_COLS]
y_mo2_tw = np.expm1(mo2[TARGET_COL])
X_va2_tw = va2[FEATURE_COLS]
y_va2_tw = np.expm1(va2[TARGET_COL].values)

print('Building Tweedie Fold 2 datasets...')
dtrain2_tw = lgb.Dataset(X_tr2_tw, label=y_tr2_tw, feature_name=list(FEATURE_COLS), free_raw_data=False)
dtrain2_tw.construct()
dmon2_tw   = lgb.Dataset(X_mo2_tw, label=y_mo2_tw, reference=dtrain2_tw, free_raw_data=False)
dmon2_tw.construct()
print('Datasets ready.')
print()

# ── Fold 1 best params (Trial 3, unit-RMSE 3.71) ──────────────
TWEEDIE_FOLD1_BEST = {
    'num_leaves':             225,
    'learning_rate':          0.02237,
    'feature_fraction':       0.9343,
    'bagging_fraction':       0.8182,
    'bagging_freq':           2,
    'min_child_samples':      47,
    'reg_alpha':              3.712,
    'reg_lambda':             2.656,
    'tweedie_variance_power': 1.069,
}


def make_objective_tweedie_f2(dtrain, dmon, X_val, y_val_units):
    def objective(trial):
        params = {
            # Fold 1 best 225, winners 111-262, high leaves lost → 150-300
            'num_leaves':             trial.suggest_int('num_leaves', 150, 300),
            # Fold 1 best 0.022, winners 0.010-0.034 → 0.010-0.040
            'learning_rate':          trial.suggest_float('learning_rate', 0.010, 0.040, log=True),
            # Fold 1 best 0.93, wide spread → keep full range
            'feature_fraction':       trial.suggest_float('feature_fraction', 0.50, 1.0),
            # Fold 1 winners 0.72-0.92 → 0.70-0.95
            'bagging_fraction':       trial.suggest_float('bagging_fraction', 0.70, 0.95),
            'bagging_freq':           trial.suggest_int('bagging_freq', 1, 7),
            # Fold 1 best 47, winners 43-90 → 30-90
            'min_child_samples':      trial.suggest_int('min_child_samples', 30, 90),
            # Fold 1 best 3.71, winners 2.0-4.8 → 2.0-5.0
            'reg_alpha':              trial.suggest_float('reg_alpha', 2.0, 5.0),
            # Fold 1 best 2.66, winners 1.2-3.3 → 1.0-4.0
            'reg_lambda':             trial.suggest_float('reg_lambda', 1.0, 4.0),
            # Fold 1 best 1.07, tvp=1.9 was disaster → 1.0-1.5
            'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.0, 1.5),
            'objective':              'tweedie',
            'metric':                 'tweedie',
            'verbosity':              -1,
            'random_state':           42,
            'num_threads':            -1,
        }

        model = lgb.train(
            params=params,
            train_set=dtrain,
            num_boost_round=2000,
            valid_sets=[dmon],
            callbacks=[
                lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        trial.set_user_attr('best_trees', model.best_iteration)
        y_pred_units = model.predict(X_val)
        mask         = y_val_units > 0
        rmse         = np.sqrt(mean_squared_error(y_val_units[mask], y_pred_units[mask]))
        return rmse

    return objective


print('Tweedie Fold 2 — Optuna tuning started...')
print(f'  {get_gpu_stats()}')
print()

study_tw_f2 = optuna.create_study(direction='minimize')
study_tw_f2.enqueue_trial(TWEEDIE_FOLD1_BEST)

study_tw_f2.optimize(
    make_objective_tweedie_f2(dtrain2_tw, dmon2_tw, X_va2_tw, y_va2_tw),
    n_trials=25,
    callbacks=[optuna_callback_tw],
)

print()
print(f'Tweedie Fold 2 best unit-RMSE : {study_tw_f2.best_value:.2f}')
print()
print('Best params:')
for k, v in study_tw_f2.best_params.items():
    print(f'  {k:<26} {v}')

Building Tweedie Fold 2 datasets...
Datasets ready.

Tweedie Fold 2 — Optuna tuning started...
  GPU: 51°C | 0 % util | 169 MiB / 6144 MiB

  Trial   0 | unit-RMSE: 3.11 | best: 3.11 | trees=1404 | leaves=225 lr=0.022 ff=0.93 bf=0.82 mcs= 47 tvp=1.07 α=3.71 λ=2.66
  GPU: 59°C | 0 % util | 169 MiB / 6144 MiB
  Trial   1 | unit-RMSE: 3.10 | best: 3.10 | trees=1172 | leaves=224 lr=0.017 ff=0.86 bf=0.86 mcs= 50 tvp=1.02 α=4.78 λ=1.94
  Trial   2 | unit-RMSE: 3.12 | best: 3.10 | trees=1997 | leaves=211 lr=0.012 ff=0.64 bf=0.92 mcs= 82 tvp=1.40 α=4.91 λ=2.90
  Trial   3 | unit-RMSE: 3.11 | best: 3.10 | trees=1990 | leaves=295 lr=0.011 ff=0.65 bf=0.87 mcs= 80 tvp=1.00 α=4.62 λ=2.73
  Trial   4 | unit-RMSE: 3.11 | best: 3.10 | trees=1319 | leaves=229 lr=0.013 ff=0.97 bf=0.92 mcs= 46 tvp=1.38 α=3.02 λ=2.72
  Trial   5 | unit-RMSE: 3.12 | best: 3.10 | trees= 546 | leaves=166 lr=0.026 ff=0.98 bf=0.74 mcs= 57 tvp=1.48 α=3.70 λ=1.38
  GPU: 59°C | 0 % util | 169 MiB / 6144 MiB
  Trial   6 | unit-RMS

[W 2026-06-02 23:07:35,856] Trial 17 failed with parameters: {'num_leaves': 274, 'learning_rate': 0.010756248393693154, 'feature_fraction': 0.6979800091351775, 'bagging_fraction': 0.792385438324845, 'bagging_freq': 5, 'min_child_samples': 54, 'reg_alpha': 3.21254836998207, 'reg_lambda': 1.5444573513271709, 'tweedie_variance_power': 1.1532832302434457} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\Zach\AppData\Local\Temp\ipykernel_17460\3561360350.py", line 63, in objective
    model = lgb.train(
        params=params,
    ...<6 lines>...
        ],
    )
  File "c:\Users\Zach\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\engine.py", line 322, in train
    booster.update(fobj=fobj)
    ~~~~~~~~~~~~~~^^^^^^^^^^^
  File "c:\Users\Zach\AppData\Loca

KeyboardInterrupt: 

In [22]:
BEST_PARAMS_LGBM = {
    'num_leaves':        508,
    'learning_rate':     0.02897,
    'feature_fraction':  0.5833,
    'bagging_fraction':  0.9495,
    'bagging_freq':      1,
    'min_child_samples': 68,
    'reg_alpha':         4.3495,
    'reg_lambda':        1.3520,
    'objective':         'regression',
    'metric':            'rmse',
    'verbosity':         -1,
    'random_state':      42,
    'num_threads':       -1,
}

print(f'BEST_PARAMS_LGBM frozen. Optuna best log-RMSE: 0.5765')
print()
print('Frozen params:')
for k, v in BEST_PARAMS_LGBM.items():
    print(f'  {k:<22} {v}')

BEST_PARAMS_LGBM frozen. Optuna best log-RMSE: 0.5765

Frozen params:
  num_leaves             508
  learning_rate          0.02897
  feature_fraction       0.5833
  bagging_fraction       0.9495
  bagging_freq           1
  min_child_samples      68
  reg_alpha              4.3495
  reg_lambda             1.352
  objective              regression
  metric                 rmse
  verbosity              -1
  random_state           42
  num_threads            -1


In [7]:
# ── BEST_PARAMS_TWEEDIE — hardcoded from Trial 1 (unit-RMSE 3.10) ──
BEST_PARAMS_TWEEDIE = {
    'num_leaves':             224,
    'learning_rate':          0.017,
    'feature_fraction':       0.86,
    'bagging_fraction':       0.86,
    'bagging_freq':           2,
    'min_child_samples':      50,
    'reg_alpha':              4.78,
    'reg_lambda':             1.94,
    'tweedie_variance_power': 1.02,
    'objective':              'tweedie',
    'metric':                 'tweedie',
    'verbosity':              -1,
    'random_state':           42,
    'num_threads':            -1,
}

print(f'BEST_PARAMS_TWEEDIE frozen. Optuna best unit-RMSE: 3.10')
print()
print('Frozen params:')
for k, v in BEST_PARAMS_TWEEDIE.items():
    print(f'  {k:<26} {v}')

BEST_PARAMS_TWEEDIE frozen. Optuna best unit-RMSE: 3.10

Frozen params:
  num_leaves                 224
  learning_rate              0.017
  feature_fraction           0.86
  bagging_fraction           0.86
  bagging_freq               2
  min_child_samples          50
  reg_alpha                  4.78
  reg_lambda                 1.94
  tweedie_variance_power     1.02
  objective                  tweedie
  metric                     tweedie
  verbosity                  -1
  random_state               42
  num_threads                -1


### Section 6: Fold 2 Retrain with Frozen LightGBM Params

In [23]:
assert BEST_PARAMS_LGBM is not None, \
    'BEST_PARAMS_LGBM is None — Section 5 (Optuna) must complete first.'

dtrain2  = lgb.Dataset(X_tr2, label=y_tr2, feature_name=list(FEATURE_COLS), free_raw_data=False)
dmonitor2 = lgb.Dataset(X_mo2, label=y_mo2, reference=dtrain2,               free_raw_data=False)

print('Training full Fold 2 LightGBM model with BEST_PARAMS_LGBM...')
print(f'{get_gpu_stats()}')
print()

t0 = time.time()
model_lgbm = lgb.train(
    params=BEST_PARAMS_LGBM,
    train_set=dtrain2,
    num_boost_round=2000,
    valid_sets=[dmonitor2],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)
elapsed = time.time() - t0

print(f'\nDone in {elapsed:.1f}s  |  best n_estimators: {model_lgbm.best_iteration}')
print(f'{get_gpu_stats()}')
print()

# Sanity check — must replicate Optuna result
y_pred_raw     = model_lgbm.predict(X_va2)
mask_check     = y_va2 > 0
rmse_check     = np.sqrt(((y_va2[mask_check] - y_pred_raw[mask_check])**2).mean())
print(f'Sanity check log-RMSE : {rmse_check:.4f}')
print(f'Optuna best was       : {study_fold2.best_value:.4f}')
delta_check = rmse_check - study_fold2.best_value
print(f'Delta (expect ≈0)     : {delta_check:+.4f}')
if abs(delta_check) < 0.002:
    print('Replication confirmed. ✓')
else:
    print('⚠ Delta exceeds tolerance — check monitor set and early_stopping_rounds.')

model_lgbm.save_model(f'{MODELS_DIR}/lgbm_model_fold2.txt')
print(f'\nModel saved: {MODELS_DIR}/lgbm_model_fold2.txt')

Training full Fold 2 LightGBM model with BEST_PARAMS_LGBM...
GPU: 52°C | 0 % util | 0 MiB / 6144 MiB

[100]	valid_0's rmse: 0.497297
[200]	valid_0's rmse: 0.493003
[300]	valid_0's rmse: 0.491929
[400]	valid_0's rmse: 0.491389
[500]	valid_0's rmse: 0.491057
[600]	valid_0's rmse: 0.490837
[700]	valid_0's rmse: 0.490687
[800]	valid_0's rmse: 0.490572
[900]	valid_0's rmse: 0.490471
[1000]	valid_0's rmse: 0.490375
[1100]	valid_0's rmse: 0.49031
[1200]	valid_0's rmse: 0.490267
[1300]	valid_0's rmse: 0.490238
[1400]	valid_0's rmse: 0.490204
[1500]	valid_0's rmse: 0.490184
[1600]	valid_0's rmse: 0.49015
[1700]	valid_0's rmse: 0.490122
[1800]	valid_0's rmse: 0.490099
[1900]	valid_0's rmse: 0.49007
[2000]	valid_0's rmse: 0.490045

Done in 781.7s  |  best n_estimators: 1996
GPU: 59°C | 0 % util | 0 MiB / 6144 MiB

Sanity check log-RMSE : 0.5766
Optuna best was       : 0.5765
Delta (expect ≈0)     : +0.0001
Replication confirmed. ✓

Model saved: ../data/processed/models/lgbm_model_fold2.txt


In [8]:
assert BEST_PARAMS_TWEEDIE is not None, \
    'BEST_PARAMS_TWEEDIE is None — Tweedie Optuna must complete first.'

dtrain2_tw_full  = lgb.Dataset(X_tr2_tw, label=y_tr2_tw, feature_name=list(FEATURE_COLS), free_raw_data=False)
dmonitor2_tw     = lgb.Dataset(X_mo2_tw, label=y_mo2_tw, reference=dtrain2_tw_full, free_raw_data=False)

print('Training full Fold 2 Tweedie model with BEST_PARAMS_TWEEDIE...')
print(f'{get_gpu_stats()}')
print()

t0 = time.time()
model_tweedie = lgb.train(
    params=BEST_PARAMS_TWEEDIE,
    train_set=dtrain2_tw_full,
    num_boost_round=2000,
    valid_sets=[dmonitor2_tw],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
        lgb.log_evaluation(period=100),
    ],
)
elapsed = time.time() - t0

print(f'\nDone in {elapsed:.1f}s  |  best n_estimators: {model_tweedie.best_iteration}')
print(f'{get_gpu_stats()}')
print()

# Sanity check — replicate Optuna result
y_pred_tw_raw  = model_tweedie.predict(X_va2_tw)
mask_check     = y_va2_tw > 0
rmse_check     = np.sqrt(mean_squared_error(y_va2_tw[mask_check], y_pred_tw_raw[mask_check]))
print(f'Sanity check unit-RMSE : {rmse_check:.2f}')
print(f'Optuna best was        : {study_tw_f2.best_value:.2f}')
delta_check = rmse_check - study_tw_f2.best_value
print(f'Delta (expect ≈0)      : {delta_check:+.4f}')
if abs(delta_check) < 1.0:
    print('Replication confirmed. ✓')
else:
    print('⚠ Delta exceeds tolerance — check monitor set and early_stopping_rounds.')

model_tweedie.save_model(f'{MODELS_DIR}/tweedie_model_fold2.txt')
print(f'\nModel saved: {MODELS_DIR}/tweedie_model_fold2.txt')

Training full Fold 2 Tweedie model with BEST_PARAMS_TWEEDIE...
GPU: 53°C | 0 % util | 169 MiB / 6144 MiB

[100]	valid_0's tweedie: 76.9257
[200]	valid_0's tweedie: 76.8531
[300]	valid_0's tweedie: 76.8418
[400]	valid_0's tweedie: 76.8374
[500]	valid_0's tweedie: 76.8352
[600]	valid_0's tweedie: 76.834
[700]	valid_0's tweedie: 76.8332
[800]	valid_0's tweedie: 76.8327
[900]	valid_0's tweedie: 76.8323
[1000]	valid_0's tweedie: 76.832
[1100]	valid_0's tweedie: 76.8316
[1200]	valid_0's tweedie: 76.8315
[1300]	valid_0's tweedie: 76.8313
[1400]	valid_0's tweedie: 76.8312

Done in 721.4s  |  best n_estimators: 1440
GPU: 57°C | 0 % util | 169 MiB / 6144 MiB

Sanity check unit-RMSE : 3.10
Optuna best was        : 3.10
Delta (expect ≈0)      : -0.0020
Replication confirmed. ✓

Model saved: ../data/processed/models/tweedie_model_fold2.txt


### Section 6b: Comparison Between RMSE and Tweedie

In [24]:
# ═══════════════════════════════════════════════════════════════
# FINAL COMPARISON — LightGBM RMSE vs LightGBM Tweedie
# ═══════════════════════════════════════════════════════════════

# ── RMSE model: bias correction diagnostics ───────────────────
train_resid = y_tr2.values - model_lgbm.predict(X_tr2)
sigma2      = float(np.var(train_resid))

y_true_units           = np.expm1(y_va2)
y_pred_units           = np.expm1(y_pred_raw)
y_pred_units_corrected = np.expm1(y_pred_raw + sigma2 / 2)
nonzero_mask           = y_true_units > 0

ratio_raw       = y_pred_units[nonzero_mask].sum()           / y_true_units[nonzero_mask].sum()
ratio_corrected = y_pred_units_corrected[nonzero_mask].sum() / y_true_units[nonzero_mask].sum()

print('Retransformation bias correction (RMSE model):')
print(f'  sigma2                       : {sigma2:.4f}')
print(f'  additive correction (σ²/2)   : {sigma2/2:.4f}')
print(f'  Standard prediction ratio    : {ratio_raw:.4f}')
print(f'  Bias-corrected ratio         : {ratio_corrected:.4f}')
print(f'  Target                       : 1.0000')
print(f'  XGBoost v2 reference         : ~0.613  (~38.7% underprediction)')
print()

# ── Tweedie model: calibration diagnostics ────────────────────
y_pred_tw_units = model_tweedie.predict(X_va2_tw)
y_true_units_tw = y_va2_tw
nonzero_mask_tw = y_true_units_tw > 0

ratio_tw = y_pred_tw_units[nonzero_mask_tw].sum() / y_true_units_tw[nonzero_mask_tw].sum()

print('Aggregate calibration check (Tweedie model):')
print(f'  Prediction ratio             : {ratio_tw:.4f}')
print(f'  Target                       : 1.0000')
print()

# ── Log-space metrics for RMSE model ──────────────────────────
r_raw = eval_log_scale(y_va2, y_pred_raw, 'LightGBM RMSE')

print('=' * 60)
print('Log-space Performance (RMSE model only)')
print('=' * 60)
print(f'  {"Metric":<18} {"XGB v2 (05c)":>12} {"LightGBM RMSE":>14}')
print(f'  {"-"*46}')
print(f'  {"log-RMSE":<18} {0.5764:>12.4f} {r_raw["log_rmse"]:>14.4f}')
print(f'  {"log-MAE":<18} {0.4707:>12.4f} {r_raw["log_mae"]:>14.4f}')
print(f'  {"Bias":<18} {-0.3750:>+12.4f} {r_raw["bias"]:>+14.4f}')
print()

# ── Head-to-head in unit space — raw uncorrected for both ─────
def model_metrics(y_true, y_pred, mask):
    rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    bias = y_pred[mask].sum() / y_true[mask].sum()
    return rmse, mape, bias

rmse_lgbm,    mape_lgbm,    bias_lgbm    = model_metrics(y_true_units,    y_pred_units,    nonzero_mask)
rmse_tweedie, mape_tweedie, bias_tweedie = model_metrics(y_true_units_tw, y_pred_tw_units, nonzero_mask_tw)

print('=' * 65)
print('HEAD-TO-HEAD — Fold 2 Validation (unit space, raw uncorrected)')
print('=' * 65)
print(f'  {"Metric":<18} {"XGB v2 (05c)":>12} {"LGBM RMSE":>12} {"LGBM Tweedie":>14}')
print(f'  {"-"*58}')
print(f'  {"unit-RMSE":<18} {"—":>12} {rmse_lgbm:>12.2f} {rmse_tweedie:>14.2f}')
print(f'  {"MAPE %":<18} {"—":>12} {mape_lgbm:>12.1f} {mape_tweedie:>14.1f}')
print(f'  {"Demand ratio":<18} {"~0.613":>12} {bias_lgbm:>12.4f} {bias_tweedie:>14.4f}')
print()

# ── Winner ─────────────────────────────────────────────────────
if rmse_lgbm <= rmse_tweedie:
    print('→ Winner: LightGBM RMSE — continue with model_lgbm')
    print('  Apply sigma2 bias correction in Section 7.')
else:
    print('→ Winner: LightGBM Tweedie — continue with model_tweedie')
    print('  No bias correction needed — Tweedie predicts in unit space directly.')

Retransformation bias correction (RMSE model):
  sigma2                       : 0.2325
  additive correction (σ²/2)   : 0.1163
  Standard prediction ratio    : 0.6113
  Bias-corrected ratio         : 0.7235
  Target                       : 1.0000
  XGBoost v2 reference         : ~0.613  (~38.7% underprediction)

Aggregate calibration check (Tweedie model):
  Prediction ratio             : 0.7760
  Target                       : 1.0000

LightGBM RMSE  [non-zero rows: 3,902,589]
  log-RMSE: 0.5766
  log-MAE:  0.4709
  Bias:     -0.3762  (+ = overpredict, − = underpredict)

Log-space Performance (RMSE model only)
  Metric             XGB v2 (05c)  LightGBM RMSE
  ----------------------------------------------
  log-RMSE                 0.5764         0.5766
  log-MAE                  0.4707         0.4709
  Bias                    -0.3750        -0.3762

HEAD-TO-HEAD — Fold 2 Validation (unit space, raw uncorrected)
  Metric             XGB v2 (05c)    LGBM RMSE   LGBM Tweedie
  ---------

## Winner Selection: LightGBM Tweedie

**Tweedie wins the Fold 2 head-to-head over LightGBM RMSE.**

The primary comparison metric for individual series forecasting is unit-space
RMSE — the error in actual demand units, which is what the Streamlit app will
display and what inventory decisions are based on.

| Metric          | LGBM RMSE | LGBM Tweedie | Winner   |
|-----------------|-----------|--------------|----------|
| unit-RMSE       | 3.39      | 3.10         | Tweedie  |
| MAPE %          | 57.7%     | 56.6%        | Tweedie  |
| Demand ratio    | 0.611     | 0.776        | Tweedie  |
| log-RMSE        | 0.5766    | n/a          | —        |

**Why Tweedie is better for this use case:**

1. **No retransformation bias.** The RMSE model trains on log(demand) and
   predicts in log space — back-transforming with expm1 introduces systematic
   underprediction (demand ratio 0.611). Tweedie predicts directly in unit
   space and has no this problem by design.

2. **Better demand ratio without calibration.** Tweedie achieves 0.776 demand
   ratio raw vs 0.611 for RMSE. The isotonic calibrator from 05c was shown to
   overcorrect individual series (577% MAPE on high-zero series) — Tweedie
   avoids this failure mode entirely.

3. **Designed for this distribution.** Tweedie loss is the statistically
   correct objective for right-skewed, zero-inflated count data. Retail demand
   at the daily SKU level is exactly this distribution.

4. **Simpler inference pipeline.** No sigma2 correction, no log retransformation.
   Predictions come out in unit space ready to use.

**Note on log-RMSE:** The plan's original primary criterion was log-RMSE.
Tweedie cannot be evaluated on log-RMSE because it never log-transforms the
target. This is not a weakness — it is a deliberate architectural choice.
The unit-space comparison is the correct criterion when the end goal is
individual series forecasting, not log-space model selection.

**Tweedie proceeds to Section 7.** A Tweedie-specific isotonic calibrator
is built and evaluated here on Fold 2 to determine whether calibration
improves Tweedie predictions before committing to it at Fold 3.

### Section 7: Post-Hoc Inference Rules and Prediction Variants

In [26]:
# ═══════════════════════════════════════════════════════════════
# Section 7 — Post-Hoc Inference Rules (Tweedie)
# ═══════════════════════════════════════════════════════════════

# ── Variant A: Raw Tweedie predictions ────────────────────────
# model_tweedie.predict() returns unit-space predictions directly
# No expm1 needed — Tweedie never log-transformed the target
y_pred_tw_raw = model_tweedie.predict(X_va2_tw)

# ── Variant B: Holiday suppression ────────────────────────────
y_pred_tw_suppressed = y_pred_tw_raw.copy()
closed_mask_tw       = va2['is_closed_holiday'].astype(bool)
y_pred_tw_suppressed[closed_mask_tw] = 0.0
print(f'Holiday suppression: {closed_mask_tw.sum():,} rows zeroed.')

# ── Variant C: Holiday suppression + Tweedie isotonic calibration ──────────
# Build a fresh isotonic calibrator for Tweedie using the same method as 05c
# but fit on Tweedie's own residuals — the 05c calibrator was fit on
# XGBoost log-space residuals and does not apply here.

# Step 1: Per-series zero rate from TRAINING data only
train_zero_rate_tw = (tr2.groupby('id')['units_sold']
                         .apply(lambda x: (x == 0).mean())
                         .rename('zero_rate'))

# Step 2: Per-series mean residual on val (suppressed predictions)
# Residuals are in unit space: actual_units - predicted_units
preds_tw_df = va2[['id']].copy()
preds_tw_df['yhat']     = y_pred_tw_suppressed
preds_tw_df['true']     = y_va2_tw  # already in unit space
preds_tw_df['residual'] = y_pred_tw_suppressed - y_va2_tw
preds_tw_df['nonzero']  = y_va2_tw > 0

series_residuals_tw = (preds_tw_df[preds_tw_df['nonzero']]
                       .groupby('id')['residual']
                       .mean()
                       .rename('mean_residual'))

# Step 3: Fit isotonic regression on zero_rate vs mean_residual
calib_df_tw = train_zero_rate_tw.to_frame().join(series_residuals_tw, how='inner').dropna()

print(f'Calibration fit: {len(calib_df_tw):,} series with zero rate + residual data')
print(f'Zero rate range : {calib_df_tw["zero_rate"].min():.3f} → {calib_df_tw["zero_rate"].max():.3f}')
print(f'Residual range  : {calib_df_tw["mean_residual"].min():.3f} → {calib_df_tw["mean_residual"].max():.3f}')
print()

ir_tw = IsotonicRegression(increasing=False, out_of_bounds='clip')
ir_tw.fit(calib_df_tw['zero_rate'].values, calib_df_tw['mean_residual'].values)

# Step 4: Apply corrections per row
id_col_tw         = va2['id'].values
zero_rate_dict_tw = train_zero_rate_tw.to_dict()
zr_per_row_tw     = np.array([zero_rate_dict_tw.get(i, 0.0) for i in id_col_tw])
corrections_tw    = -ir_tw.predict(zr_per_row_tw)

y_pred_tw_calibrated = y_pred_tw_suppressed + corrections_tw
y_pred_tw_calibrated[closed_mask_tw] = 0.0
print(f'Calibration applied: {len(ir_tw.X_thresholds_)} isotonic thresholds.')
print(f'Correction range   : {corrections_tw.min():.4f} to {corrections_tw.max():.4f}')
print()

# ── Three-variant preview in unit space ───────────────────────
print('Three-variant preview (non-zero actual rows, unit space):')
y_true_tw   = y_va2_tw
mask_nz_tw  = y_true_tw > 0

for label, y_pred in [
    ('A — Raw',        y_pred_tw_raw),
    ('B — Suppressed', y_pred_tw_suppressed),
    ('C — Calibrated', y_pred_tw_calibrated),
]:
    rmse  = np.sqrt(mean_squared_error(y_true_tw[mask_nz_tw], y_pred[mask_nz_tw]))
    bias  = float((y_pred[mask_nz_tw] - y_true_tw[mask_nz_tw]).mean())
    ratio = y_pred[mask_nz_tw].sum() / y_true_tw[mask_nz_tw].sum()
    print(f'  Variant {label}  unit-RMSE={rmse:.4f}  bias={bias:+.4f}  demand_ratio={ratio:.4f}')
print()

# ── OOS calibration check: fit on first half, evaluate on second half ──────
val_dates_tw = np.sort(va2['date'].unique())
mid_date_tw  = val_dates_tw[len(val_dates_tw) // 2]

print(f'OOS calibration check:')
print(f'  First half  : {val_dates_tw[0]} → {mid_date_tw}')
print(f'  Second half : {mid_date_tw} → {val_dates_tw[-1]}')
print()

mask_h1 = va2['date'].values <= mid_date_tw
mask_h2 = va2['date'].values >  mid_date_tw

# Refit calibrator on first half only
preds_h1 = pd.DataFrame({
    'id':       va2.loc[mask_h1, 'id'].values,
    'yhat':     y_pred_tw_suppressed[mask_h1],
    'true':     y_va2_tw[mask_h1],
    'residual': y_pred_tw_suppressed[mask_h1] - y_va2_tw[mask_h1],
    'nonzero':  y_va2_tw[mask_h1] > 0,
})

series_resid_h1 = (preds_h1[preds_h1['nonzero']]
                   .groupby('id')['residual']
                   .mean()
                   .rename('mean_residual'))

calib_h1 = (train_zero_rate_tw.to_frame()
                               .join(series_resid_h1, how='inner')
                               .dropna()
                               .reset_index())

print(f'  H1 calibration fit: {len(calib_h1):,} series')

ir_tw_h1 = IsotonicRegression(increasing=False, out_of_bounds='clip')
ir_tw_h1.fit(calib_h1['zero_rate'].values, calib_h1['mean_residual'].values)

# Apply to second half
zr_h2      = np.array([zero_rate_dict_tw.get(i, 0.0) for i in va2.loc[mask_h2, 'id'].values])
corr_h2    = -ir_tw_h1.predict(zr_h2)
y_h2_supp  = y_pred_tw_suppressed[mask_h2]
y_h2_calib = y_h2_supp + corr_h2
y_h2_calib[closed_mask_tw[mask_h2]] = 0.0
y_h2_true  = y_va2_tw[mask_h2]
mask_h2_nz = y_h2_true > 0

rmse_h2_raw   = np.sqrt(mean_squared_error(y_h2_true[mask_h2_nz], y_h2_supp[mask_h2_nz]))
rmse_h2_calib = np.sqrt(mean_squared_error(y_h2_true[mask_h2_nz], y_h2_calib[mask_h2_nz]))
ratio_h2_raw  = y_h2_supp[mask_h2_nz].sum()  / y_h2_true[mask_h2_nz].sum()
ratio_h2_cal  = y_h2_calib[mask_h2_nz].sum() / y_h2_true[mask_h2_nz].sum()

print()
print('OOS second-half results (calibrator fit on first half only):')
print(f'  {"Variant":<20} {"unit-RMSE":>12} {"Demand ratio":>14}')
print(f'  {"-"*48}')
print(f'  {"B — Suppressed":<20} {rmse_h2_raw:>12.4f} {ratio_h2_raw:>14.4f}')
print(f'  {"C — Calibrated":<20} {rmse_h2_calib:>12.4f} {ratio_h2_cal:>14.4f}')
print()

# ── Calibration gate ───────────────────────────────────────────
bias_improvement     = abs(ratio_h2_raw - 1.0) - abs(ratio_h2_cal - 1.0)
rmse_delta           = rmse_h2_calib - rmse_h2_raw
bias_improvement_pct = bias_improvement / abs(ratio_h2_raw - 1.0) * 100

print('Calibration gate (same criteria as 05d):')
print(f'  Bias improvement (demand ratio closer to 1.0) : {bias_improvement_pct:.1f}%  (gate: ≥20%)')
print(f'  RMSE delta (negative = better)                : {rmse_delta:+.4f}       (gate: <+0.5)')
print()

TWEEDIE_CALIBRATION_ADOPTED = (bias_improvement_pct >= 20.0) and (rmse_delta < 0.5)

if TWEEDIE_CALIBRATION_ADOPTED:
    print('✓ Calibration ADOPTED for Tweedie — passes both gates.')
    print('  Use Variant C (suppressed + calibrated) going forward.')
    TWEEDIE_APPROVED_VARIANT = 'calibrated'
else:
    print('✗ Calibration REJECTED for Tweedie — failed gate.')
    print('  Use Variant B (suppressed only) going forward.')
    TWEEDIE_APPROVED_VARIANT = 'suppressed'

print()
print(f'  TWEEDIE_CALIBRATION_ADOPTED : {TWEEDIE_CALIBRATION_ADOPTED}')
print(f'  TWEEDIE_APPROVED_VARIANT    : {TWEEDIE_APPROVED_VARIANT}')

Holiday suppression: 51,003 rows zeroed.
Calibration fit: 29,374 series with zero rate + residual data
Zero rate range : 0.001 → 1.000
Residual range  : -17.721 → 1.176

Calibration applied: 89 isotonic thresholds.
Correction range   : 0.3706 to 0.9393

Three-variant preview (non-zero actual rows, unit space):
  Variant A — Raw  unit-RMSE=3.1025  bias=-0.7496  demand_ratio=0.7760
  Variant B — Suppressed  unit-RMSE=3.1195  bias=-0.7530  demand_ratio=0.7750
  Variant C — Calibrated  unit-RMSE=3.0262  bias=+0.0319  demand_ratio=1.0095

OOS calibration check:
  First half  : 2014-02-01T00:00:00.000000000 → 2014-08-02T00:00:00.000000000
  Second half : 2014-08-02T00:00:00.000000000 → 2015-01-31T00:00:00.000000000

  H1 calibration fit: 27,708 series

OOS second-half results (calibrator fit on first half only):
  Variant                 unit-RMSE   Demand ratio
  ------------------------------------------------
  B — Suppressed             3.0155         0.7649
  C — Calibrated             


Out-of-sample validation of the Tweedie isotonic calibration layer. The
calibrator is fit on Fold 2 val **first half** (Feb 2014 → Aug 2014) and
evaluated on the **second half** (Aug 2014 → Jan 2015) — a genuine temporal
out-of-sample test. This confirms whether the per-series bias correction
generalizes across time or merely memorizes the val window.

| | unit-RMSE | Demand Ratio |
|---|---|---|
| Suppressed only (no calib) | 3.0155 | 0.7649 |
| Calibrated — OOS | 2.9182 | 1.0093 |
| Calibrated — in-sample | 3.0262 | 1.0095 |

OOS unit-RMSE (2.9182) is **better** than in-sample (3.0262) — the opposite
of overfitting. The per-series underprediction pattern is stable across both
halves of the val window, confirming the correction generalizes.

Demand ratio improvement: 0.7649 → 1.0093 on genuinely unseen data — a
96.1% bias reduction. The isotonic curve learned from the first half transfers
cleanly to the second half, validating the choice of a monotone non-parametric
fit.

RMSE delta: −0.0973 on the OOS half — calibration is actively improving
point accuracy, not just correcting aggregate bias. This is a stronger result
than the XGBoost v2 calibration in 05c, where OOS RMSE improved modestly.
For Tweedie the correction is larger and cleaner because Tweedie residuals
in unit space have a more stable zero-rate vs bias relationship than
log-space residuals.

**Calibration gate results:**
- Bias improvement: 96.1% ✓ (gate: ≥20%)
- RMSE delta: −0.0973 ✓ (gate: <+0.5)

**Conclusion:** Calibration ADOPTED for Tweedie. ✓  
Variant C (holiday suppression + isotonic calibration) is the approved
pipeline variant. The Tweedie calibrator will be re-fit from Fold 3 training
data at final evaluation — method identical, training window one year longer.

### Section 8: Save All Outputs

In [29]:
import os
import pickle

print('Saving 06 outputs...')
print()

# ── Predictions parquet (all three variants — Tweedie unit space) ──────────
predictions_tweedie = va2[['id', 'date', 'units_sold']].copy()
predictions_tweedie['date']            = pd.to_datetime(predictions_tweedie['date'])
predictions_tweedie['yhat_raw']        = y_pred_tw_raw
predictions_tweedie['yhat_suppressed'] = y_pred_tw_suppressed
predictions_tweedie['yhat_calibrated'] = y_pred_tw_calibrated
predictions_tweedie['true_units']      = y_va2_tw

# Leakage assertion before saving
assert predictions_tweedie['date'].max() < pd.Timestamp('2015-02-01'), \
    'LEAKAGE: predictions exceed Fold 2 boundary'
assert predictions_tweedie['date'].min() == pd.Timestamp('2014-02-01'), \
    'Val start mismatch'
assert predictions_tweedie[['yhat_raw', 'yhat_suppressed', 'yhat_calibrated']].isna().sum().sum() == 0, \
    'NaN found in predictions'

preds_path = f'{PREDICTIONS_DIR}/tweedie_predictions_fold2.parquet'
predictions_tweedie.to_parquet(preds_path, index=False)
print(f'✓ tweedie_predictions_fold2.parquet   ({len(predictions_tweedie):,} rows)')

# ── Model ──────────────────────────────────────────────────────────────────
model_path = f'{MODELS_DIR}/tweedie_model_fold2.txt'
model_tweedie.save_model(model_path)
print(f'✓ tweedie_model_fold2.txt')

# ── Best params ────────────────────────────────────────────────────────────
params_path = f'{CALIBRATION_DIR}/tweedie_best_params.pkl'
with open(params_path, 'wb') as f:
    pickle.dump(BEST_PARAMS_TWEEDIE, f)
print(f'✓ tweedie_best_params.pkl')

# ── Isotonic calibrator ────────────────────────────────────────────────────
calib_path = f'{CALIBRATION_DIR}/tweedie_isotonic_calibrator_fold2.pkl'
with open(calib_path, 'wb') as f:
    pickle.dump(ir_tw, f)
print(f'✓ tweedie_isotonic_calibrator_fold2.pkl')

# ── Zero rate lookup ───────────────────────────────────────────────────────
zr_path = f'{CALIBRATION_DIR}/tweedie_train_zero_rate_fold2.pkl'
with open(zr_path, 'wb') as f:
    pickle.dump(train_zero_rate_tw, f)
print(f'✓ tweedie_train_zero_rate_fold2.pkl')

# ── Pipeline decisions ─────────────────────────────────────────────────────
pipeline_decisions_tw = {
    'CALIBRATION_ADOPTED':   TWEEDIE_CALIBRATION_ADOPTED,
    'APPROVED_VARIANT':      TWEEDIE_APPROVED_VARIANT,
    'winner_model':          'tweedie',
    'winner_notebook':       '06_lightgbm_demand',
    'fold2_unit_rmse':       3.0262,
    'fold2_demand_ratio':    1.0095,
}
decisions_path = f'{CALIBRATION_DIR}/tweedie_pipeline_decisions_fold2.pkl'
with open(decisions_path, 'wb') as f:
    pickle.dump(pipeline_decisions_tw, f)
print(f'✓ tweedie_pipeline_decisions_fold2.pkl')

# ── Confirm all files exist ────────────────────────────────────────────────
print()
print('─' * 55)
print('File confirmation:')
print('─' * 55)
outputs = [
    preds_path,
    model_path,
    params_path,
    calib_path,
    zr_path,
    decisions_path,
]
all_present = True
for path in outputs:
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path) / 1024 / 1024:.1f} MB' if exists else '—'
    print(f'  {"✓" if exists else "✗"}  {os.path.basename(path):<45} {size}')
    if not exists:
        all_present = False

print()
if all_present:
    print('All outputs saved. 06 complete.')
    print()
    print('Winner : LightGBM Tweedie')
    print(f'  Approved variant              : {TWEEDIE_APPROVED_VARIANT}')
    print(f'  Fold 2 unit-RMSE (calibrated) : 3.0262')
    print(f'  Fold 2 demand ratio           : 1.0095')
    print(f'  Calibration adopted           : {TWEEDIE_CALIBRATION_ADOPTED}')
    print()
    print('Next: 07_fold3_final_evaluation.ipynb')
    print('  — Load tweedie_best_params.pkl')
    print('  — Load tweedie_pipeline_decisions_fold2.pkl')
    print('  — Retrain Tweedie on full Fold 3 training data')
    print('  — Re-fit isotonic calibrator from Fold 3 training data')
    print('  — Compare vs XGBoost v2 on Fold 3 val window')
else:
    print('⚠ Some outputs missing — check paths above.')

Saving 06 outputs...

✓ tweedie_predictions_fold2.parquet   (9,004,868 rows)
✓ tweedie_model_fold2.txt
✓ tweedie_best_params.pkl
✓ tweedie_isotonic_calibrator_fold2.pkl
✓ tweedie_train_zero_rate_fold2.pkl
✓ tweedie_pipeline_decisions_fold2.pkl

───────────────────────────────────────────────────────
File confirmation:
───────────────────────────────────────────────────────
  ✓  tweedie_predictions_fold2.parquet             226.4 MB
  ✓  tweedie_model_fold2.txt                       34.5 MB
  ✓  tweedie_best_params.pkl                       0.0 MB
  ✓  tweedie_isotonic_calibrator_fold2.pkl         0.0 MB
  ✓  tweedie_train_zero_rate_fold2.pkl             1.3 MB
  ✓  tweedie_pipeline_decisions_fold2.pkl          0.0 MB

All outputs saved. 06 complete.

Winner : LightGBM Tweedie
  Approved variant              : calibrated
  Fold 2 unit-RMSE (calibrated) : 3.0262
  Fold 2 demand ratio           : 1.0095
  Calibration adopted           : True

Next: 07_fold3_final_evaluation.ipynb
  — Load